# Naive Bayes

**INDE 577 / CMOR 438 — Rice University**  
**Instructor:** Randy R. Davila, PhD

---

## Overview

Naive Bayes is a **probabilistic classifier** based on Bayes' theorem with the "naive" assumption that features are **conditionally independent** given the class label. Despite this strong assumption, it often performs surprisingly well in practice.

## Mathematical Background

### Bayes' Theorem

$$P(y \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid y) \cdot P(y)}{P(\mathbf{x})}$$

### Naive Independence Assumption

$$P(\mathbf{x} \mid y) = \prod_{j=1}^{d} P(x_j \mid y)$$

### Classification Rule (MAP estimate)

$$\hat{y} = \arg\max_k P(y = k) \prod_{j=1}^{d} P(x_j \mid y = k)$$

In log-space (to avoid underflow):

$$\hat{y} = \arg\max_k \left[ \log P(y = k) + \sum_{j=1}^{d} \log P(x_j \mid y = k) \right]$$

### Gaussian Naive Bayes

For continuous features, model each $P(x_j \mid y = k)$ as a Gaussian:

$$P(x_j \mid y = k) = \frac{1}{\sqrt{2\pi \sigma_{jk}^2}} \exp\left(-\frac{(x_j - \mu_{jk})^2}{2\sigma_{jk}^2}\right)$$

Learned parameters: **class mean** $\mu_{jk}$ and **class variance** $\sigma_{jk}^2$ for each feature $j$ and class $k$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer, load_wine
from sklearn.naive_bayes import GaussianNB as SklearnNB
import warnings
warnings.filterwarnings('ignore')

from rice_ml import GaussianNaiveBayes
from rice_ml.processing.preprocessing import StandardScaler, train_test_split
from rice_ml.processing.metrics import accuracy_score, classification_report

print("Libraries loaded!")
np.random.seed(42)

## 1. Iris Dataset

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
target_names = iris.target_names
feature_names = iris.feature_names

X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

nb_iris = GaussianNaiveBayes()
nb_iris.fit(X_tr_i, y_tr_i)
y_pred_i = nb_iris.predict(X_te_i)

print("=== Gaussian Naive Bayes — Iris ===")
print(classification_report(y_te_i, y_pred_i, target_names=list(target_names)))

In [ ]:
# Visualize learned Gaussian distributions for petal length
feature_idx = 2  # petal length
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = ['#e74c3c', '#3498db', '#2ecc71']
x_range = np.linspace(X_iris[:, feature_idx].min() - 0.5, X_iris[:, feature_idx].max() + 0.5, 300)

for class_idx, (name, color) in enumerate(zip(target_names, colors)):
    # Learned parameters
    mu = nb_iris.theta_[class_idx, feature_idx]
    sigma2 = nb_iris.sigma_[class_idx, feature_idx]
    
    # Data histogram
    mask = y_iris == class_idx
    ax.hist(X_iris[mask, feature_idx], bins=15, alpha=0.3, color=color, density=True, edgecolor='k', lw=0.3)
    
    # Gaussian fit
    gaussian = (1 / np.sqrt(2 * np.pi * sigma2)) * np.exp(-(x_range - mu)**2 / (2 * sigma2))
    ax.plot(x_range, gaussian, color=color, linewidth=2.5, label=f'{name} (μ={mu:.2f}, σ={np.sqrt(sigma2):.2f})')

ax.set_xlabel('Petal Length (cm)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Learned Gaussian Distributions\n(Petal Length per Class)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Decision boundaries on 2D feature space
ax2 = axes[1]
X_2f = X_iris[:, 2:4]  # petal features
nb_2d = GaussianNaiveBayes()
nb_2d.fit(X_2f, y_iris)

x_min, x_max = X_2f[:, 0].min() - 0.3, X_2f[:, 0].max() + 0.3
y_min, y_max = X_2f[:, 1].min() - 0.3, X_2f[:, 1].max() + 0.3
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
Z = nb_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

ax2.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn', levels=[-0.5, 0.5, 1.5, 2.5])
for i, (name, color) in enumerate(zip(target_names, colors)):
    mask = y_iris == i
    ax2.scatter(X_2f[mask, 0], X_2f[mask, 1], c=color, s=40, alpha=0.8, edgecolors='k', lw=0.4, label=name)
ax2.set_xlabel('Petal Length (cm)', fontsize=11)
ax2.set_ylabel('Petal Width (cm)', fontsize=11)
ax2.set_title('Naive Bayes Decision Boundary\n(Petal Features)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('naive_bayes_iris.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Breast Cancer Classification

In [ ]:
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
target_names_bc = data.target_names

X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

nb_bc = GaussianNaiveBayes()
nb_bc.fit(X_tr, y_tr)
y_pred_bc = nb_bc.predict(X_te)
proba = nb_bc.predict_proba(X_te)

sk_nb = SklearnNB()
sk_nb.fit(X_tr, y_tr)

our_acc = accuracy_score(y_te, y_pred_bc)
sk_acc = accuracy_score(y_te, sk_nb.predict(X_te))

print("=== Gaussian Naive Bayes — Breast Cancer ===")
print(classification_report(y_te, y_pred_bc, target_names=list(target_names_bc)))
print(f"rice_ml accuracy: {our_acc:.4f}")
print(f"sklearn accuracy: {sk_acc:.4f}")

In [ ]:
# Posterior probability distribution
fig, ax = plt.subplots(figsize=(9, 5))
benign_proba = proba[:, 1]  # P(benign | x)

ax.hist(benign_proba[y_te == 0], bins=25, alpha=0.7, color='#e74c3c', label='Malignant (true)', edgecolor='k', lw=0.4, density=True)
ax.hist(benign_proba[y_te == 1], bins=25, alpha=0.7, color='#2ecc71', label='Benign (true)', edgecolor='k', lw=0.4, density=True)
ax.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Decision threshold')
ax.set_xlabel('P(Benign | x)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Posterior Probability Distribution\n(Naive Bayes — Breast Cancer)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('naive_bayes_proba.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Wine Dataset — Multi-Class

In [ ]:
wine = load_wine()
X_w, y_w = wine.data, wine.target
X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(X_w, y_w, test_size=0.2, random_state=42)

nb_w = GaussianNaiveBayes()
nb_w.fit(X_tr_w, y_tr_w)
print("Wine Dataset — Naive Bayes:")
print(classification_report(y_te_w, nb_w.predict(X_te_w), target_names=[f'Class {i}' for i in range(3)]))
print(f"Accuracy: {accuracy_score(y_te_w, nb_w.predict(X_te_w)):.4f}")

## Summary

| Property | Value |
|---|---|
| Type | Probabilistic generative classifier |
| Assumption | Conditional independence of features |
| Distribution | Gaussian (for continuous features) |
| Training | MLE for per-class mean and variance |
| Speed | Very fast — $O(n \cdot d \cdot K)$ training |

**Key Takeaways:**
- Naive Bayes is **extremely fast** and works well with small datasets
- The independence assumption is rarely true but often sufficient in practice
- Performs well on **text classification** (with Multinomial NB) and high-dimensional data
- Provides **probability estimates** in addition to class predictions
- Works well when features are approximately Gaussian and classes are well-separated